# Music Annotation Project
## Notebook 74C: Ranked Collar Genuine-Print Loss Extract

**Runtime: CPU. Read-only.** This is a small follow-on to the completed combined Notebook 74 audit.

The combined Notebook 74 outputs already contain the pixel-level evidence needed for most of the analysis, but they do **not** contain the requested connected-component ranking with bounding boxes. This notebook derives that extract without rerunning any correction model.

### Output
For the paired-development cohort only, rank connected regions of **clean-supported genuine printed ink that was whitened in Ring 1 or Ring 2**. Each row contains:

- page and source group;
- ring (`RING1` or `RING2`);
- connected-component ID;
- component area and bounding box;
- true-print pixels whitened;
- whitening mass and maximum whitening;
- annotation-distance band and numeric distance summaries;
- route distribution;
- a page/component severity rank.

It also creates review-friendly contact sheets showing observed, clean, corrected, authority geometry and a focused collar-loss overlay.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Resolve the completed combined Notebook 74 run

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB74_ROOT=ROOT/'notebook74_combined_mask_annotation_distance_audit'
SOURCE_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook74c_ranked_collar_print_loss'
TOP_COMPONENTS=100
CONTACT_COMPONENTS=40
CROP_MARGIN=90

REQUIRED=['inputs/combined_roster.csv','tables/page_summary.csv','tables/pixel_parts','audit/input_resolution_and_geometry.csv']
def latest_complete(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
SOURCE=Path(SOURCE_RUN_OVERRIDE) if SOURCE_RUN_OVERRIDE else latest_complete(NB74_ROOT)
if SOURCE is None:raise RuntimeError('No complete combined Notebook 74 run found.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook74c_collar_extract'
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';VISUAL=OUT/'visuals';CROPS=OUT/'crops';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,VISUAL,CROPS,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Source:',SOURCE);print('Output:',OUT)

## 2. Verify paired pixel parts and source roster

In [ ]:
def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def read_gray(p):
    a=cv2.imread(str(p),0);require(a is not None,f'Unreadable image: {p}');return a

roster=pd.read_csv(SOURCE/'inputs/combined_roster.csv',dtype=str).fillna('')
paired=roster[roster.population=='PAIRED_DEVELOPMENT'].copy()
require(len(paired)>0,'No paired-development rows in combined roster.')
parts=SOURCE/'tables/pixel_parts'
missing=[r.page_key for _,r in paired.iterrows() if not (parts/f'{r.page_key}.parquet').exists()]
require(not missing,f'Missing paired pixel parts: {missing}')
required_cols={'page_key','page','population','source_group','y','x','authority_band','route','observed','signed_change','whitened_8plus','truth_print_posthoc','distance_to_truth_annotation_posthoc','truth_distance_band'}
for _,r in paired.iterrows():
    cols=set(pd.read_parquet(parts/f'{r.page_key}.parquet').columns)
    require(required_cols<=cols,f'{r.page_key}: missing columns {sorted(required_cols-cols)}')
print('Paired pages:',len(paired))

## 3. Build connected collar-loss components

In [ ]:
rows=[]
for _,r in paired.iterrows():
    part=pd.read_parquet(parts/f'{r.page_key}.parquet')
    q=part[part.authority_band.isin(['RING1','RING2']) & part.truth_print_posthoc.astype(bool) & part.whitened_8plus.astype(bool)].copy()
    if q.empty:continue
    H=int(part.y.max())+1;W=int(part.x.max())+1
    for ring in ['RING1','RING2']:
        z=q[q.authority_band==ring].copy()
        if z.empty:continue
        mask=np.zeros((H,W),np.uint8);mask[z.y.to_numpy(int),z.x.to_numpy(int)]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(mask,8)
        labels=lab[z.y.to_numpy(int),z.x.to_numpy(int)];z['collar_component_id']=labels
        for cid,g in z[z.collar_component_id>0].groupby('collar_component_id'):
            st=stats[int(cid)];change=np.maximum(pd.to_numeric(g.signed_change,errors='coerce').to_numpy(float),0);dist=pd.to_numeric(g.distance_to_truth_annotation_posthoc,errors='coerce').to_numpy(float)
            route_counts=g.route.value_counts().to_dict();dominant=max(route_counts,key=route_counts.get) if route_counts else 'UNKNOWN'
            distance_band=g.truth_distance_band.astype(str).value_counts().idxmax()
            rows.append({'page_key':r.page_key,'page':r.page,'source_group':r.source_group,'ring':ring,'component_id':int(cid),'x':int(st[cv2.CC_STAT_LEFT]),'y':int(st[cv2.CC_STAT_TOP]),'w':int(st[cv2.CC_STAT_WIDTH]),'h':int(st[cv2.CC_STAT_HEIGHT]),'component_pixels':int(st[cv2.CC_STAT_AREA]),'true_print_pixels_whitened':len(g),'whitening_mass':float(change.sum()),'mean_whitening':float(change.mean()),'max_whitening':float(change.max()),'mean_distance_to_true_annotation':float(np.nanmean(dist)),'min_distance_to_true_annotation':float(np.nanmin(dist)),'p90_distance_to_true_annotation':float(np.nanpercentile(dist,90)),'dominant_distance_band':distance_band,'dominant_route':dominant,'route_counts_json':json.dumps(route_counts,sort_keys=True)})
    del part,q;gc.collect()
components=pd.DataFrame(rows)
require(len(components)>0,'No collar genuine-print whitening components found.')
components=components.sort_values(['whitening_mass','true_print_pixels_whitened','max_whitening'],ascending=False).reset_index(drop=True)
components.insert(0,'severity_rank',np.arange(1,len(components)+1))
components.to_csv(TABLES/'ranked_collar_genuine_print_loss_components.csv',index=False)
components.head(TOP_COMPONENTS).to_csv(TABLES/'top_collar_genuine_print_loss_components.csv',index=False)
print(components.head(30).to_string(index=False))

## 4. Page/source/ring summaries

In [ ]:
by_page=(components.groupby(['page_key','page','source_group','ring'],as_index=False).agg(components=('component_id','size'),true_print_pixels_whitened=('true_print_pixels_whitened','sum'),whitening_mass=('whitening_mass','sum'),largest_component=('true_print_pixels_whitened','max'),max_whitening=('max_whitening','max'),mean_distance_to_true_annotation=('mean_distance_to_true_annotation','mean'))).sort_values('whitening_mass',ascending=False)
by_page.to_csv(TABLES/'collar_loss_by_page_and_ring.csv',index=False)
by_source=(components.groupby(['source_group','ring'],as_index=False).agg(pages=('page_key','nunique'),components=('component_id','size'),true_print_pixels_whitened=('true_print_pixels_whitened','sum'),whitening_mass=('whitening_mass','sum'),largest_component=('true_print_pixels_whitened','max'))).sort_values(['ring','whitening_mass'],ascending=[True,False])
by_source.to_csv(TABLES/'collar_loss_by_source_group_and_ring.csv',index=False)
by_distance=(components.groupby(['ring','dominant_distance_band'],as_index=False).agg(components=('component_id','size'),pages=('page_key','nunique'),true_print_pixels_whitened=('true_print_pixels_whitened','sum'),whitening_mass=('whitening_mass','sum')))
by_distance.to_csv(TABLES/'collar_loss_by_distance_band.csv',index=False)
print(by_source.to_string(index=False))

## 5. Visual review sheets

In [ ]:
def paths_for(page_key):
    p=SOURCE/'pages'/page_key
    return p

def circle_overlay(gray,row):
    rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB);x0=int(row.x);y0=int(row.y);x1=x0+int(row.w);y1=y0+int(row.h)
    cv2.rectangle(rgb,(x0,y0),(x1,y1),(255,0,0),4);cv2.putText(rgb,f'{row.ring} rank {int(row.severity_rank)}',(x0,max(18,y0-8)),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,0,0),2,cv2.LINE_AA);return rgb

review=components.head(CONTACT_COMPONENTS)
fig,axes=plt.subplots(len(review),4,figsize=(18,max(6,4.6*len(review))),squeeze=False)
for i,(_,r) in enumerate(review.iterrows()):
    p=paths_for(r.page_key);obs=read_gray(p/'observed.png');cor=read_gray(p/'corrected.png');clean=read_gray(p/'clean.png') if (p/'clean.png').exists() else None
    x0=max(0,int(r.x)-CROP_MARGIN);y0=max(0,int(r.y)-CROP_MARGIN);x1=min(obs.shape[1],int(r.x+r.w)+CROP_MARGIN);y1=min(obs.shape[0],int(r.y+r.h)+CROP_MARGIN)
    authority=cv2.imread(str(p/'authority_overlay.png'),cv2.IMREAD_COLOR) if (p/'authority_overlay.png').exists() else None
    if authority is not None:authority=cv2.cvtColor(authority,cv2.COLOR_BGR2RGB)
    marked=circle_overlay(obs,r)
    ims=[marked[y0:y1,x0:x1],clean[y0:y1,x0:x1] if clean is not None else np.full_like(obs[y0:y1,x0:x1],255),cor[y0:y1,x0:x1],authority[y0:y1,x0:x1] if authority is not None else marked[y0:y1,x0:x1]]
    titles=['observed + ranked box','clean truth','corrected','authority geometry']
    for j,(im,title) in enumerate(zip(ims,titles)):
        axes[i,j].imshow(im,cmap='gray' if im.ndim==2 else None);axes[i,j].set_title(f'#{int(r.severity_rank)} {r.page_key} {r.ring}\n{title}\nprint={int(r.true_print_pixels_whitened)} mass={r.whitening_mass:.0f} dist={r.mean_distance_to_true_annotation:.1f}');axes[i,j].axis('off')
plt.tight_layout();plt.savefig(VISUAL/'top_collar_genuine_print_loss_components.png',dpi=145,bbox_inches='tight');plt.close(fig)

# Compact page-level sheet for top pages.
top_pages=by_page.drop_duplicates('page_key').head(20)
fig,axes=plt.subplots(len(top_pages),4,figsize=(18,max(6,5*len(top_pages))),squeeze=False)
for i,(_,r) in enumerate(top_pages.iterrows()):
    p=paths_for(r.page_key);files=[('observed.png','observed'),('clean.png','clean'),('corrected.png','corrected'),('truth_overlay_posthoc.png','truth ownership/change')]
    for j,(fn,title) in enumerate(files):
        im=cv2.imread(str(p/fn),cv2.IMREAD_UNCHANGED)
        if im is not None and im.ndim==3:im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
        axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key}\n{title}');axes[i,j].axis('off')
plt.tight_layout();plt.savefig(VISUAL/'top_collar_loss_pages.png',dpi=125,bbox_inches='tight');plt.close(fig)
print('Visuals:',VISUAL)

## 6. Decision extract and upload package

In [ ]:
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'source_notebook74_run':str(SOURCE),'paired_pages':int(paired.page_key.nunique()),'collar_components':len(components),'pages_with_collar_genuine_print_loss':int(components.page_key.nunique()),'ring1_true_print_pixels_whitened':int(components.loc[components.ring=='RING1','true_print_pixels_whitened'].sum()),'ring2_true_print_pixels_whitened':int(components.loc[components.ring=='RING2','true_print_pixels_whitened'].sum()),'largest_component_pixels':int(components.true_print_pixels_whitened.max()),'largest_component_whitening_mass':float(components.whitening_mass.max()),'note':'Connected-component ranking is diagnostic evidence. It does not alter the frozen correction policy.'}
(OUT/'collar_loss_extract_summary.json').write_text(json.dumps(summary,indent=2))
lines=['NOTEBOOK 74C — RANKED COLLAR GENUINE-PRINT LOSS EXTRACT','',f'Paired pages: {summary["paired_pages"]}',f'Pages with collar loss: {summary["pages_with_collar_genuine_print_loss"]}',f'Connected collar-loss components: {summary["collar_components"]}',f'Ring 1 true-print pixels whitened: {summary["ring1_true_print_pixels_whitened"]}',f'Ring 2 true-print pixels whitened: {summary["ring2_true_print_pixels_whitened"]}',f'Largest component: {summary["largest_component_pixels"]} true-print pixels',f'Largest component whitening mass: {summary["largest_component_whitening_mass"]:.0f}','','Use ranked_collar_genuine_print_loss_components.csv and top_collar_genuine_print_loss_components.png together.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'collar_loss_extract_summary.json',TABLES/'ranked_collar_genuine_print_loss_components.csv',TABLES/'top_collar_genuine_print_loss_components.csv',TABLES/'collar_loss_by_page_and_ring.csv',TABLES/'collar_loss_by_source_group_and_ring.csv',TABLES/'collar_loss_by_distance_band.csv',VISUAL/'top_collar_genuine_print_loss_components.png',VISUAL/'top_collar_loss_pages.png']
for p in required:shutil.copy2(p,UPLOAD/p.name)
rows=[]
for p in UPLOAD.iterdir():
    if p.is_file():rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 74C ranked paired-development collar genuine-print loss extract. Start with executive_summary.txt, ranked_collar_genuine_print_loss_components.csv and the two visual sheets.\n')
with zipfile.ZipFile(UPLOAD/'notebook74c_collar_loss_extract.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('Bundle:',UPLOAD/'notebook74c_collar_loss_extract.zip')